# COMP7103 Lab 10 · Extract fields from job ads, then retrieve

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/doubledaibo/COMP7103B/blob/main/labs/lab10/Lab10_extract_retrieve.ipynb)

Two short tasks on prepared data. First, a job board wants a table from 50 job ads: you extract six fields with **rules** (regular expressions) and with an **LLM** (large language model), check every value, and score both against a gold table. Second, a public library wants its search box to find the right help page or book: you compare **BM25** (keyword) and **dense** (embedding) retrieval on 10 labelled queries.

| Time | What you do |
|---|---|
| 5 min | Brief |
| 15 min | Extraction: rules vs LLM |
| 15 min | Retrieval: BM25 vs dense |
| 10 min | Share-out |

This lab is ungraded practice for the final exam. Data: synthetic job ads and a made-up library, written for this lab (CC BY 4.0); prepared LLM output from Qwen2.5-1.5B-Instruct and embeddings from all-MiniLM-L6-v2 (both Apache 2.0). Details in `data/SOURCE.txt`.

**Using this notebook in Colab.** Choose *File → Save a copy in Drive* first, and work in your copy. Run a cell with **Shift + Enter**. Run the cells from top to bottom. Where you see `___`, replace it with your own code, then run the cell again.

## 0 · Setup

This cell loads the 50 job ads, the gold table, the library records, the 10 queries and the embeddings computed in advance. Nothing to install.

In [ ]:
import os, re, json, warnings, urllib.request
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import CountVectorizer
warnings.filterwarnings("ignore")
pd.set_option("display.max_colwidth", 120)

# A blank (___) that you have not filled in stops with: name '___' is not defined.
# (The next two lines make sure of that. You do not need to read or change them.)
get_ipython().displayhook.cache_size = 0
globals().pop("___", None)

REPO = "https://raw.githubusercontent.com/doubledaibo/COMP7103B/main/labs/lab10/data/"

# get(name) gives the path of a data file: the copy next to the notebook if there is one, otherwise a fresh
# download from the course GitHub. You do not need to read how it works.
def get(name):
    """Use the local file if it is there (Lab 10 downloaded from GitHub), otherwise download it."""
    for path in [name, os.path.join("data", name),]:
        if os.path.exists(path):
            return path
    os.makedirs("data", exist_ok=True)
    try:
        urllib.request.urlretrieve(REPO + name, os.path.join("data", name))
    except Exception:
        raise SystemExit(f"Could not download {name}. Upload it with the folder icon on the left, "
                         "or download the Lab 10 from GitHub: https://github.com/doubledaibo/COMP7103B/tree/main/labs/lab10")
    return os.path.join("data", name)

FIELDS = ["title", "company", "salary", "district", "experience", "contract"]
DISTRICTS = ["Central and Western", "Wan Chai", "Eastern", "Southern", "Yau Tsim Mong", "Sham Shui Po", "Kowloon City",
             "Wong Tai Sin", "Kwun Tong", "Kwai Tsing", "Tsuen Wan", "Tuen Mun", "Yuen Long", "North", "Tai Po",
             "Sha Tin", "Sai Kung", "Islands"]

def as_value(x):
    """A table cell as a Python value: empty -> None, 12000.0 -> 12000."""
    if x is None or (isinstance(x, float) and np.isnan(x)):
        return None                          # an empty cell
    if isinstance(x, float) and x.is_integer():
        return int(x)                        # 12000.0 -> 12000
    return x

ads = pd.read_csv(get("job_ads.csv"))
gold_table = pd.read_csv(get("gold.csv"))
gold = []                                    # one dictionary per ad: field -> the correct value
for i in range(len(gold_table)):
    values = {}
    for f in FIELDS:
        values[f] = as_value(gold_table[f].iloc[i])
    gold.append(values)

corpus = pd.read_csv(get("corpus.csv"))
queries = pd.read_csv(get("queries.csv"))
doc_emb = np.load(get("doc_emb.npy"))        # one embedding (384 numbers) per library record
query_emb = np.load(get("query_emb.npy"))    # one embedding per query
print(len(ads), "job ads ·", len(corpus), "library records ·", len(queries), "queries · embeddings", doc_emb.shape, query_emb.shape)

In [ ]:
ads.head(3)

## 1 · Rules: finish two regular expressions (4 min)

Information extraction fills a **fixed schema** from free text: the same fields, types and allowed values for every ad.

| Field | Type | Allowed values |
|---|---|---|
| `title` | text | any job title |
| `company` | text | a company name |
| `salary` | integer | HK\$ per month, 8,000 to 200,000 (the lower end of a range) |
| `district` | category | one of Hong Kong's 18 districts |
| `experience` | integer | minimum years, 0 to 40 |
| `contract` | category | full-time, part-time, contract |

An empty value (`None`) means "not stated". It is a valid answer: an hourly rate or "salary negotiable" gives salary `None`.

The rules for title, company, district and contract are given. Each rule is a regular expression; the part in brackets `( )` is a *group*, the piece of text the rule keeps.

In [ ]:
# re.compile(pattern, flags) makes a rule. The pieces used below:
#   ( )      a group: the piece of text the rule keeps     .+    any text (one or more characters)
#   \s       a space                                       \s*   any number of spaces
#   ^ and $  the start and the end of a line (re.M)        \b    a word boundary
#   a|b      a or b                                        re.I  ignore upper and lower case
RULES = {
    "title":    re.compile(r"^Position:\s*(.+)$", re.M),                       # the text after "Position:"
    # the text after "Company:", or (|) capitalised words that end in " Ltd"
    "company":  re.compile(r"^Company:\s*(.+)$|\b([A-Z][A-Za-z]+(?: [A-Z][A-Za-z]+)* Ltd)\b", re.M),
    "district": re.compile(r"\b(" + "|".join(DISTRICTS) + r")\b", re.I),       # one of the 18 names
    "contract": re.compile(r"\b(full[- ]time|part[- ]time|contract)\b", re.I),   # [- ] = a hyphen or a space
}

def rule_value(field, text):
    """The value the rule for this field finds in the ad, or None."""
    m = RULES[field].search(text)            # the first match in the ad, or None
    if m is None:
        return None
    if field == "company":                   # the company rule has two groups: use the one that matched
        if m.group(1) is not None:
            return m.group(1).strip()
        return m.group(2).strip()
    if field == "salary":                    # "28,000" -> 28000; "65" and "k" -> 65000
        number = int(m.group(1).replace(",", ""))
        if m.group(2):                       # group 2 is the "k"
            number = number * 1000
        return number
    if field == "experience":
        return int(m.group(1))
    if field == "district":                  # write the name as in the list: "wan chai" -> "Wan Chai"
        for d in DISTRICTS:
            if d.lower() == m.group(1).lower():
                return d
    if field == "contract":                  # "Full time" -> "full-time"
        return m.group(1).lower().replace(" ", "-")
    return m.group(1).strip()

Finish the two missing rules.

- **Salary**: `HK$`, an optional space, then the number with its commas as group 1 (a digit, then any digits or commas), then an optional `k` as group 2. `[\d,]*` means "any number of digits or commas".
- **Experience**: the number as group 1, an optional `+` (as in 2+) or range end (as in 3-5), then the word *year* or *years*, then up to two words and *experience*. `s?` means "an optional s".

In [ ]:
# Salary: "HK$" (\$ is a plain $), an optional space (\s?), YOUR GROUP 1, then an optional "k" as group 2
RULES["salary"] = re.compile(r"HK\$\s?" + ___ + r"(\s?k\b)?", re.I)
# Experience: the number as group 1 (\d+ = one or more digits), an optional "+" or range end ("-5"), YOUR WORD,
# an optional apostrophe, an optional "of", up to two more words, then "experience"
RULES["experience"] = re.compile(r"(\d+)\+?\s*(?:-\s*\d+\s*)?" + ___ + r"['’]?\s+(?:of\s+)?(?:[a-z]+\s+){0,2}experience", re.I)

print(rule_value("salary", "HK$65k to 80k"), rule_value("salary", "HK$ 19,000/month"))       # expect 65000 19000
print(rule_value("experience", "1 year of experience"), rule_value("experience", "3-5 years' IT support experience"))  # expect 1 3

Run all six rules on all 50 ads.

In [ ]:
rules = []                                # one dictionary per ad: field -> the value the rule finds
for text in ads["text"]:
    values = {}
    for f in FIELDS:
        values[f] = rule_value(f, text)
    rules.append(values)
pd.DataFrame(rules, index=ads["ad_id"], dtype=object).head(5)    # the first 5 ads as a table

## 2 · The LLM: paste the prepared message (4 min)

1. Run the next cell. It prints a message: the schema and ads 1 to 5.
2. Paste the message into any chat LLM you use. The ads are synthetic; there is no personal data.
3. Paste the LLM's whole answer between the triple quotes in the cell after.

No LLM available? Use the ready-made answer (see the comment in that cell). It comes from Qwen2.5-1.5B-Instruct, a small open model. The other 45 ads come with prepared output from the same model and the same message.

In [ ]:
print(open(get("llm_prompt.txt")).read())

In [ ]:
llm_reply = """
paste the LLM's answer here
"""
# No LLM? Use the ready-made answer instead:
# llm_reply = open(get("llm_fallback.txt")).read()
print(llm_reply)

The parser keeps every `{...}` in the answer as one ad's JSON. Then the 45 prepared answers are added: `llm` holds the LLM's values for all 50 ads.

In [ ]:
def json_blocks(text):
    """Every piece of the text from a "{" to the next "}" (with no other "{" in between)."""
    blocks = []
    start = text.find("{")                   # .find gives the position, or -1 if it is not there
    while start != -1:
        end = text.find("}", start)
        if end == -1:
            break                            # no closing "}": stop
        inner = text.find("{", start + 1)
        if inner != -1 and inner < end:
            start = inner                    # another "{" comes first: start again from there
            continue
        blocks.append(text[start:end + 1])
        start = text.find("{", end)          # look for the next block
    return blocks

def parse_llm(reply, ad_ids):
    """Every {...} in the reply as a dictionary, matched to an ad by its "ad" number (or else by its position)."""
    objs = []
    for block in json_blocks(reply):
        try:
            objs.append(json.loads(block))   # JSON text -> a Python dictionary
        except json.JSONDecodeError:
            objs.append({})                  # not valid JSON: every field stays empty

    by_ad = {}                               # ad number (as text) -> its dictionary
    for o in objs:
        by_ad[str(o.get("ad"))] = o          # .get gives None if there is no "ad" key

    out = []
    for pos in range(len(ad_ids)):
        a = str(ad_ids[pos])
        if a in by_ad:
            o = by_ad[a]                     # found by its ad number
        elif pos < len(objs):
            o = objs[pos]                    # no ad number: take the block at the same position
        else:
            o = {}                           # missing
        values = {}
        for f in FIELDS:
            values[f] = o.get(f)             # None if the LLM gave no value
        out.append(values)

    n_bad = 0
    for o in objs:
        if o == {}:
            n_bad = n_bad + 1
    if n_bad > 0 or len(objs) < len(ad_ids):
        print(f"Warning: {len(objs)} {{...}} blocks found for {len(ad_ids)} ads, {n_bad} not valid JSON "
              "(trailing comma, // comment?). Fix llm_reply by hand.")
    return out

llm = parse_llm(llm_reply, [1, 2, 3, 4, 5])                               # ads 1 to 5: your answer
llm = llm + parse_llm(open(get("llm_output_45.jsonl")).read(), list(range(6, 51)))   # ads 6 to 50: prepared

# Ads 1 to 5 side by side: the gold value, the rules' value and the LLM's value
sources = {"gold": gold, "rules": rules, "LLM": llm}
side = []
for i in range(5):
    for name in sources:
        row = {"ad": i + 1, "source": name}
        row.update(sources[name][i])         # add the six fields
        side.append(row)
pd.DataFrame(side, dtype=object).set_index(["ad", "source"])

## 3 · Check, then score per field (7 min)

Never trust an extracted value as is. Three checks from the lecture, on every value:

- **Type**: salary and experience are integers; district is one of the 18 names; contract is one of the three values.
- **Range**: salary from HK\$8,000 to HK\$200,000 per month; experience from 0 to 40 years.
- **Value in source**: the value appears in the ad (a number among the ad's numbers; a text ignoring case and hyphens).

Fill in the blank: a number passes when it is one of the numbers in the ad.

In [ ]:
ALLOWED = ["full-time", "part-time", "contract"]

def is_int(v):
    """True for a whole number such as 28000 or 28000.0 (but not True/False, which Python also counts as numbers)."""
    return isinstance(v, (int, float)) and not isinstance(v, bool) and float(v).is_integer()

def numbers_in(text):
    """All numbers written in the ad: "28,000" -> 28000, "65k" -> 65000; "no experience" counts as 0."""
    out = []
    # a digit, then digits or commas, an optional decimal part, and an optional "k"
    for s in re.findall(r"\d[\d,]*(?:\.\d+)?(?:\s?k\b)?", text, re.I):
        number = s.lower().replace(",", "").replace("k", "").strip()
        v = float(number)
        if s.lower().endswith("k"):
            v = v * 1000
        out.append(v)
    if "no experience" in text.lower():
        out.append(0)
    return out

def simplify(text):
    """Lower case, hyphens as spaces, single spaces: "Full-Time" and "full  time" both -> "full time"."""
    return " ".join(str(text).lower().replace("-", " ").split())

# Each check returns True (passes), False (fails) or None (not checked: the value is empty)
def type_ok(f, v):
    if v is None:
        return None
    if f in ("salary", "experience"):
        return is_int(v)
    if f == "district":
        return v in DISTRICTS
    if f == "contract":
        return v in ALLOWED
    return isinstance(v, str) and len(v) > 0     # title, company: any text that is not empty

def range_ok(f, v):
    if v is None or not is_int(v):
        return None                              # only numbers have a range
    if f == "salary":
        return 8_000 <= v <= 200_000
    if f == "experience":
        return 0 <= v <= 40
    return None

def in_source(f, v, text):
    if v is None:
        return None
    if is_int(v):
        return ___
    return simplify(v) in simplify(text)     # a text value: is it in the ad (ignoring case and hyphens)?

def failures(preds):
    """How many values each check flags, per field (a value that is None is not checked)."""
    texts = list(ads["text"])
    table = {"type": {}, "range": {}, "value in source": {}}
    for f in FIELDS:
        n_type = 0
        n_range = 0
        n_source = 0
        for i in range(len(preds)):
            v = preds[i][f]
            if type_ok(f, v) == False:       # == False: None (not checked) does not count
                n_type = n_type + 1
            if range_ok(f, v) == False:
                n_range = n_range + 1
            if in_source(f, v, texts[i]) == False:
                n_source = n_source + 1
        table["type"][f] = n_type
        table["range"][f] = n_range
        table["value in source"][f] = n_source
    return pd.DataFrame(table).T             # one row per check, one column per field

flags = pd.concat({"rules": failures(rules), "LLM": failures(llm)})    # the two tables on top of each other
flags["all fields"] = flags.sum(axis=1)
flags

Now score each field against the gold table, as in the lecture. **TP**: a value was extracted and it equals the gold value. **FP**: a value was extracted, but it is wrong or the ad states none. **FN**: the ad states a value, but the extracted one is missing or wrong (a wrong value counts as FP and as FN).

$$P = \frac{TP}{TP + FP} \qquad R = \frac{TP}{TP + FN} \qquad F_1 = \frac{2PR}{P + R}$$

Fill in the F1 line.

In [ ]:
def same(p, g):
    """Equal, ignoring case and spaces at the ends for text."""
    if p is None or g is None:
        return p is g
    if isinstance(p, str):
        return p.strip().lower() == str(g).strip().lower()
    return p == g

def score_field(f, preds):
    """TP, FP, FN, precision, recall and F1 of one field over the 50 ads."""
    tp = 0
    fp = 0
    fn = 0
    for i in range(len(gold)):
        p = preds[i][f]                      # the extracted value
        g = gold[i][f]                       # the correct value
        if p is not None and same(p, g):
            tp = tp + 1                      # extracted and right
        else:
            if p is not None:
                fp = fp + 1                  # extracted, but wrong
            if g is not None:
                fn = fn + 1                  # the ad states a value, but it was not found right
    if tp + fp > 0:
        P = tp / (tp + fp)
    else:
        P = np.nan                           # nothing extracted: precision is undefined
    if tp + fn > 0:
        R = tp / (tp + fn)
    else:
        R = np.nan
    F1 = ___
    return {"TP": tp, "FP": fp, "FN": fn, "P": round(P, 2), "R": round(R, 2), "F1": round(F1, 2)}

tables = {}                                  # method -> a table with one row per field
methods = {"rules": rules, "LLM": llm}
for name in methods:
    rows = {}
    for f in FIELDS:
        rows[f] = score_field(f, methods[name])
    tables[name] = pd.DataFrame.from_dict(rows, orient="index")     # orient="index": one row per field
scores = pd.concat(tables, axis=1)           # the two tables side by side
scores

In [ ]:
f1_rules = scores["rules"]["F1"]
f1_llm = scores["LLM"]["F1"]
x = np.arange(len(FIELDS))                   # 0, 1, ..., 5: one position per field
fig, ax = plt.subplots(figsize=(8, 3.2))
ax.bar(x - 0.2, f1_rules, 0.4, facecolor="none", edgecolor="grey", linewidth=1.5, label="rules")   # left bar
ax.bar(x + 0.2, f1_llm, 0.4, color="#6d4fc2", label="LLM")                                          # right bar
ax.set_xticks(x, FIELDS)
ax.set_ylim(0, 1.25)
ax.set_yticks([0, 0.25, 0.5, 0.75, 1])
ax.set_ylabel("F1")
ax.legend(loc="upper center", ncol=2, frameon=False)
ax.set_title("F1 per field on 50 job ads: rules vs LLM")
plt.tight_layout()
plt.show()

Every LLM value that differs from the gold table, or that a check flags. The summary counts them by "right?" and "flagged by"; the table below it shows only the interesting rows: errors that pass all three checks (flagged by "-") and false alarms (right, but flagged). Which errors does a check catch, and which pass all three checks? Which flags are false alarms?

In [ ]:
texts = list(ads["text"])
rows = []
for i in range(len(llm)):
    for f in FIELDS:
        v = llm[i][f]                        # the LLM's value
        g = gold[i][f]                       # the correct value
        flagged = []                         # the names of the checks that fail
        if type_ok(f, v) == False:
            flagged.append("type")
        if range_ok(f, v) == False:
            flagged.append("range")
        if in_source(f, v, texts[i]) == False:
            flagged.append("source")
        right = same(v, g)
        if not right or "type" in flagged or "source" in flagged:
            if len(flagged) > 0:
                flagged_by = ", ".join(flagged)
            else:
                flagged_by = "-"             # no check caught it
            rows.append({"ad": i + 1, "field": f, "gold": g, "LLM": v, "right?": right, "flagged by": flagged_by})
errs = pd.DataFrame(rows, dtype=object)

# How many rows of each kind
print(errs.groupby(["right?", "flagged by"]).size().to_string(), "\n")
# The interesting rows: errors that pass every check, and right values that a check flags (false alarms)
errs[(errs["flagged by"] == "-") | errs["right?"]]

## 4 · Retrieval: BM25 vs dense (15 min)

The library's search box covers 210 records: help-guide chunks (rules, fees, rooms, digital services) and catalogue records (title, author, format, call number, subject, a one-line summary). A person marked which records answer each of the 10 queries (`relevant`).

In [ ]:
print(corpus["kind"].value_counts().to_string(), "\n")      # how many records of each kind
for t in corpus["text"][:3]:                                  # the first 3 records
    print("-", t)
queries

**BM25** refines TF-IDF: extra occurrences of a term within the same record help less and less (saturation, set by $k_1$), and long records are penalized (length normalization, set by $b$). For a query $Q$ and a record $D$:

$$\text{score}(D, Q) = \sum_{t \in Q} \text{IDF}(t)\cdot\frac{f(t, D)\,(k_1 + 1)}{f(t, D) + k_1\left(1 - b + b\cdot |D| / \text{avgdl}\right)}$$

$f(t, D)$: count of term $t$ in $D$. $|D|$: length of $D$ in terms; avgdl: the average length. $\text{IDF}(t) = \ln\left(1 + \frac{N - n_t + 0.5}{n_t + 0.5}\right)$, with $N$ records and $n_t$ records containing $t$.

Terms are lower-case words and numbers, without stop words (*the, for, how, …*) and with a plural *s* removed, as in the lecture. `CountVectorizer` counts every term in every record. Fill in the lecture's values: $k_1 = 1.2$, $b = 0.75$.

In [ ]:
STOP = set("a an and are as at be by can do for from has have how i if in is it its my of on or our the their there "
           "this to was we what when where which who will with you your".split())

def toks(s):
    """Lower-case words and numbers (hyphenated ones kept), no stop words, a plural s removed."""
    out = []
    # the words: letters and digits, with at most one hyphen inside ("ill-2", "e-book")
    for w in re.findall(r"[a-z0-9]+(?:-[a-z0-9]+)?", s.lower()):
        if w in STOP:
            continue                                  # skip stop words
        if len(w) > 4 and w.endswith("ies"):
            w = w[:-3] + "y"                          # "libraries" -> "library"
        elif len(w) > 3 and w.endswith("s") and not w.endswith("ss"):
            w = w[:-1]                                # "books" -> "book" (but not "class")
        out.append(w)
    return out

cv = CountVectorizer(analyzer=toks)                   # count the terms that toks() gives
F = cv.fit_transform(corpus["text"]).toarray()        # F[d, t] = f(t, D): records x terms
N = len(F)                                            # the number of records
dl = F.sum(axis=1)                                    # |D|: the length of each record in terms
avgdl = dl.mean()                                     # the average length
n_t = (F > 0).sum(axis=0)                             # n_t: the number of records that contain term t
IDF = np.log(1 + (N - n_t + 0.5) / (n_t + 0.5))       # one IDF value per term
print(N, "records ·", F.shape[1], "terms · avgdl", round(avgdl, 1))

def bm25(query, k1=___, b=___):
    """BM25 score of every record for the query."""
    scores = np.zeros(N)                              # one score per record, starting at 0
    for t in toks(query):                             # the sum over the query terms
        if t not in cv.vocabulary_:
            continue                                  # no record contains t: it adds nothing
        j = cv.vocabulary_[t]                         # the column of term t in F
        f = F[:, j]                                   # f(t, D) for every record D
        scores = scores + IDF[j] * f * (k1 + 1) / (f + k1 * (1 - b + b * dl / avgdl))
    return scores

def ranking(scores):
    """Record ids, highest score first (equal scores in record order)."""
    order = np.argsort(-scores, kind="stable")        # positions sorted by score, highest first
    return corpus["doc_id"].values[order]

print(list(ranking(bm25("Form ILL-2"))[:5]))

**Dense retrieval** ranks records by the cosine of their embeddings (all-MiniLM-L6-v2, 384 numbers each, computed in advance for every record and every query). The embeddings have length 1, so the cosine is a dot product. Fill in: the scores of all records for query `i`.

In [ ]:
def dense(i):
    """Cosine of every record with query i (row i of query_emb)."""
    return doc_emb @ ___                              # @ is the dot product: one cosine per record

print(list(ranking(dense(0))[:5]))

**Recall@k**: of all the relevant records of a query, the share that is in the top $k$.

$$\text{Recall@}k = \frac{\text{relevant records in the top } k}{\text{all relevant records of the query}}$$

Fill in the function, then compute recall@5 for every query and the mean for each retriever.

In [ ]:
def recall_at(ranked, relevant, k=5):
    top_k = list(ranked[:k])                          # the first k record ids
    found = 0                                         # relevant records in the top k
    for doc in relevant:
        if doc in top_k:
            found = found + 1
    return ___

rows = []
for i in range(len(queries)):
    query = queries["query"][i]
    relevant = queries["relevant"][i].split()         # "D012 D044" -> ["D012", "D044"]
    ranked_bm25 = ranking(bm25(query))
    ranked_dense = ranking(dense(i))
    rows.append({"query": query, "relevant": queries["relevant"][i],
                 "BM25 recall@5": recall_at(ranked_bm25, relevant),
                 "dense recall@5": recall_at(ranked_dense, relevant),
                 "BM25 top 5": " ".join(ranked_bm25[:5]),
                 "dense top 5": " ".join(ranked_dense[:5])})
res = pd.DataFrame(rows, index=queries["query_id"])
res.loc["mean"] = ["", "", res["BM25 recall@5"].mean(), res["dense recall@5"].mean(), "", ""]   # a last row
res.round(2)

Look closer at a query: its relevant records and the top 5 of each retriever. ✓ marks a relevant record. Change `qid` to look at others.

In [ ]:
def show(qid):
    """A query, its relevant records, and the top 5 of each retriever with their scores."""
    i = list(queries["query_id"]).index(qid)          # the row of this query
    query = queries["query"][i]
    relevant = queries["relevant"][i].split()
    print(f"{qid}: '{query}' · relevant: {queries['relevant'][i]}")
    all_scores = {"BM25": bm25(query), "dense": dense(i)}
    for name in all_scores:
        scores = all_scores[name]
        ranked = ranking(scores)
        print(f"\n{name}, recall@5 = {recall_at(ranked, relevant):.2f}")
        for d in ranked[:5]:
            row = list(corpus["doc_id"]).index(d)     # the row of this record
            if d in relevant:
                mark = "✓"
            else:
                mark = " "
            print(f"  {mark} {d} {scores[row]:6.3f}  {corpus['text'][row][:95]}")

qid = "Q1"
show(qid)

Find one query where BM25 wins and one where dense wins. Say why in one sentence each: is it an exact term (a code, a number, a rare word) or a paraphrase (the same meaning in other words)?

In [ ]:
BM25_WINS = "Q?"
WHY_BM25 = "one sentence"
DENSE_WINS = "Q?"
WHY_DENSE = "one sentence"
print("BM25 wins", BM25_WINS, "·", WHY_BM25)
print("Dense wins", DENSE_WINS, "·", WHY_DENSE)

## 5 · Share-out (10 min)

- Report F1 per field (title, company, salary, district, experience, contract) for the rules and for the LLM, and the mean recall@5 for BM25 and for dense; we collect them on the class board. (The LLM column is mostly the small prepared model: 45 of the 50 ads.)
- How many values did the type, range and value-in-source checks flag for each extractor? Name one LLM error that passes all three checks.
- Your two queries: where BM25 wins, where dense wins, and why.
- Which chat LLM did you use, and did its five answers differ from the ready-made ones?